In [ ]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, random_split


In [ ]:
df = pd.read_csv("/content/role_challenge_dataset_ground_truth.csv")
df

,image_name,ofd_1_x,ofd_1_y,ofd_2_x,ofd_2_y,bpd_1_x,bpd_1_y,bpd_2_x,bpd_2_y
0,000_HC.png,361,12,339,530,481,16,664,318
1,001_HC.png,441,331,368,308,297,247,534,142
2,002_HC.png,318,374,154,406,481,158,558,215
3,003_HC.png,424,105,407,462,305,349,547,363
4,004_HC.png,300,277,611,534,53,452,494,308
...,...,...,...,...,...,...,...,...,...
617,497_HC.png,407,77,432,411,181,270,634,281
618,498_HC.png,445,75,412,458,129,243,687,275
619,499_2HC.png,432,105,411,470,194,284,647,296
620,499_HC.png,356,119,357,484,138,347,608,272


In [ ]:
!unzip -o /content/images-20260211T124919Z-1-001.zip -d /content
!unzip -o /content/masks-20260218T055803Z-1-001.zip -d /content


Archive:  /content/images-20260211T124919Z-1-001.zip
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of /content/images-20260211T124919Z-1-001.zip or
        /content/images-20260211T124919Z-1-001.zip.zip, and cannot find /content/images-20260211T124919Z-1-001.zip.ZIP, period.
Archive:  /content/masks-20260218T055803Z-1-001.zip
  inflating: /content/masks/490_HC_Annotation.png  
  inflating: /content/masks/495_2HC_Annotation.png  
  inflating: /content/masks/496_HC_Annotation.png  
  inflating: /content/masks/490_2HC_Annotation.png  
  inflating: /content/masks/483_HC_Annotation.png  
  inflating: /content/masks/489_HC_Annotation.png  
  inflating: /content/masks/487_HC_Annotation.png  
  inflating: /content/masks/482_HC_Annotation.png  


In [ ]:
!ls /content


images-20260211T124919Z-1-001.zip  role_challenge_dataset_ground_truth.csv
masks				   sample_data
masks-20260218T055803Z-1-001.zip


In [ ]:
!ls /content/images | head
!ls /content/masks | head


ls: cannot access '/content/images': No such file or directory
000_HC_Annotation.png
001_HC_Annotation.png
002_HC_Annotation.png
003_HC_Annotation.png
004_HC_Annotation.png
005_HC_Annotation.png
006_HC_Annotation.png
007_HC_Annotation.png
008_HC_Annotation.png
009_HC_Annotation.png


In [ ]:
IMAGE_ROOT = "/content/images"
MASK_ROOT  = "/content/masks"


In [ ]:
class SegmentationDataset(torch.utils.data.Dataset):
    def __init__(self, image_root, mask_root, image_size=128):
        self.image_root = image_root
        self.mask_root = mask_root
        self.image_size = image_size
        self.images = sorted(os.listdir(image_root))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image_name = self.images[idx]

        image_path = os.path.join(self.image_root, image_name)

        base_name = image_name.replace(".png", "")
        mask_name = base_name + "_Annotation.png"
        mask_path = os.path.join(self.mask_root, mask_name)

        image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        mask  = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        if image is None:
            raise FileNotFoundError(image_path)

        if mask is None:
            raise FileNotFoundError(mask_path)

        image = cv2.resize(image, (self.image_size, self.image_size))
        mask  = cv2.resize(mask,  (self.image_size, self.image_size))

        image = image.astype(np.float32) / 255.0
        mask  = mask.astype(np.float32) / 255.0

        return torch.tensor(image).unsqueeze(0), torch.tensor(mask).unsqueeze(0)


In [ ]:
dataset = SegmentationDataset(IMAGE_ROOT, MASK_ROOT)


FileNotFoundError: [Errno 2] No such file or directory: '/content/images'

In [ ]:
img, mask = dataset[0]
print(img.shape)
print(mask.shape)


NameError: name 'dataset' is not defined

In [ ]:
!ls /content/masks | head


In [ ]:
plt.imshow(img[0], cmap="gray")
plt.imshow(mask[0], alpha=0.5)
plt.title("Image + Mask Overlay")
plt.show()


In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    def __init__(self, base=32):
        super().__init__()

        self.enc1 = DoubleConv(1, base)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = DoubleConv(base, base*2)
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = DoubleConv(base*2, base*4)
        self.pool3 = nn.MaxPool2d(2)

        self.enc4 = DoubleConv(base*4, base*8)
        self.pool4 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(base*8, base*16)

        self.up4 = nn.ConvTranspose2d(base*16, base*8, 2, stride=2)
        self.dec4 = DoubleConv(base*16, base*8)

        self.up3 = nn.ConvTranspose2d(base*8, base*4, 2, stride=2)
        self.dec3 = DoubleConv(base*8, base*4)

        self.up2 = nn.ConvTranspose2d(base*4, base*2, 2, stride=2)
        self.dec2 = DoubleConv(base*4, base*2)

        self.up1 = nn.ConvTranspose2d(base*2, base, 2, stride=2)
        self.dec1 = DoubleConv(base*2, base)

        self.final = nn.Conv2d(base, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        e4 = self.enc4(self.pool3(e3))

        b = self.bottleneck(self.pool4(e4))

        d4 = self.up4(b)
        d4 = torch.cat([d4, e4], dim=1)
        d4 = self.dec4(d4)

        d3 = self.up3(d4)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.final(d1)


In [ ]:
def dice_loss(pred, target, smooth=1):
    pred = torch.sigmoid(pred)
    intersection = (pred * target).sum()
    return 1 - (2. * intersection + smooth) / (
        pred.sum() + target.sum() + smooth
    )


In [ ]:
bce = nn.BCEWithLogitsLoss()

def combined_loss(pred, target):
    return bce(pred, target) + dice_loss(pred, target)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [ ]:
from torch.utils.data import DataLoader, random_split

dataset = SegmentationDataset(IMAGE_ROOT, MASK_ROOT)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)


In [ ]:
model = UNet(base=32).to(device)


In [ ]:
img, mask = dataset[0]
print(mask.min(), mask.max())


In [ ]:
model.eval()
img, mask = dataset[0]

with torch.no_grad():
    pred = model(img.unsqueeze(0).to(device))
    pred = torch.sigmoid(pred).cpu()[0][0]

plt.imshow(pred)
plt.colorbar()
plt.show()


In [ ]:
criterion = nn.BCEWithLogitsLoss()


In [ ]:
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([5.0]).to(device))


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)


In [ ]:
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for images, masks in train_loader:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = combined_loss(outputs, masks)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")


In [ ]:
torch.save(model.state_dict(), "segmentation_model.pth")


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet(base=32).to(device)

model.load_state_dict(torch.load("segmentation_model.pth", map_location=device))

print("Previous weights loaded.")


NameError: name 'UNet' is not defined